In [25]:
import os, time, tempfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (AutoModelForImageClassification, AutoImageProcessor,
                          TrainingArguments, Trainer)
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             mean_absolute_error, confusion_matrix)

from settings.config import IMAGE_SOURCE, LABEL_SOURCE, N_CLASSES
from utils.image_utils import list_images, image_to_patches

In [26]:
EPOCHS = 1
BATCH  = 64
LR     = 1e-4
SEED   = 42
N_SUBSET_IMAGES = 10

In [34]:
MODEL_NAMES = {
    'ResNet-18':         'microsoft/resnet-18',
    'MobileNetV2':       'google/mobilenet_v2_1.0_224',
    'MobileNetV3-Small': 'google/mobilenet_v1_1.0_224',
}

In [28]:
CACHE_DIR = 'patch_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def stem_of(url):
    return os.path.splitext(os.path.basename(url))[0]

def build_cache(images):
    for url in images:
        stem = stem_of(url)
        out = os.path.join(CACHE_DIR, stem + '.npz')
        if os.path.exists(out):
            continue
        patches = np.stack(image_to_patches(url))
        labels = np.load(os.path.join(LABEL_SOURCE, stem + '.npy')).reshape(-1)
        assert len(patches) == len(labels), f'The number of cells does not match. {stem}'
        np.savez(out, patches=patches, labels=labels)
        print('Cut:', stem)

def load_cached(images):
    P, L = [], []
    for url in images:
        d = np.load(os.path.join(CACHE_DIR, stem_of(url) + '.npz'))
        P.append(d['patches']); L.append(d['labels'])
    return np.concatenate(P), np.concatenate(L)

In [29]:
subset = list(list_images(IMAGE_SOURCE))[:N_SUBSET_IMAGES]
build_cache(subset)

rng = np.random.default_rng(SEED)
rng.shuffle(subset)
n_val = max(1, int(len(subset) * 0.2))
val_imgs, train_imgs = subset[:n_val], subset[n_val:]

train_patches, train_labels = load_cached(train_imgs)
val_patches,   val_labels   = load_cached(val_imgs)
print(f'training {len(train_labels)} grid / valid {len(val_labels)} grid')

training 11856 grid / valid 2964 grid


In [30]:
class CloudDataset(Dataset):
    def __init__(self, patches, labels, processor):
        self.patches = patches
        self.labels = labels
        self.processor = processor
    def __len__(self):
        return len(self.patches)
    def __getitem__(self, i):
        out = self.processor(self.patches[i], return_tensors='pt')
        return {
            'pixel_values': out['pixel_values'][0],
            'labels': torch.tensor(int(self.labels[i])),
        }

def make_processor(hf_id):
    p = AutoImageProcessor.from_pretrained(hf_id)
    if hasattr(p, 'crop_pct'):
        p.crop_pct = 1.0
    if hasattr(p, 'do_center_crop'):
        p.do_center_crop = False
    return p

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'kappa': cohen_kappa_score(labels, preds, weights='quadratic'),
        'mae': mean_absolute_error(labels, preds),
    }

In [31]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

def size_mb(m):
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
        torch.save(m.state_dict(), f.name); s = os.path.getsize(f.name) / 1024**2
    os.remove(f.name); return s

def speed_ms(m, size=224, n=30, warmup=5):
    m.eval().to(device)
    x = torch.randn(1, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(warmup): m(x)
        if device.type == 'mps': torch.mps.synchronize()
        t = time.time()
        for _ in range(n): m(x)
        if device.type == 'mps': torch.mps.synchronize()
    return (time.time() - t) / n * 1000

In [32]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

device: mps


In [35]:
rows, cms = [], {}
for name, hf_id in MODEL_NAMES.items():
    print(f'=== {name} ({hf_id}) ===')
    processor = make_processor(hf_id)
    train_ds = CloudDataset(train_patches, train_labels, processor)
    val_ds   = CloudDataset(val_patches,   val_labels,   processor)

    model = AutoModelForImageClassification.from_pretrained(
        hf_id, num_labels=N_CLASSES, ignore_mismatched_sizes=True)

    args = TrainingArguments(
        output_dir=f'cmp_{name}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        learning_rate=LR,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=10,
        seed=SEED,
        remove_unused_columns=False,
        report_to='none'
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()

    pred = trainer.predict(val_ds)
    gts, preds = pred.label_ids, pred.predictions.argmax(-1)
    rows.append({
        'Model': name,
        'Parameters(M)': round(count_params(model) / 1e6, 2),
        'Size(MB)': round(size_mb(model), 1),
        'Speed(ms)': round(speed_ms(model), 1),
        'accuracy': round(accuracy_score(gts, preds), 4),
        'macro_f1': round(f1_score(gts, preds, average='macro'), 4),
        'kappa': round(cohen_kappa_score(gts, preds, weights='quadratic'), 4),
        'MAE': round(mean_absolute_error(gts, preds), 4),
    })
    cms[name] = confusion_matrix(gts, preds, labels=list(range(N_CLASSES)))

df = pd.DataFrame(rows)
df

=== ResNet-18 (microsoft/resnet-18) ===


Some weights of ResNetForImageClassification were not initialized from the model checkpoint at microsoft/resnet-18 and are newly initialized because the shapes did not match:
- classifier.1.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.1.weight: found shape torch.Size([1000, 512]) in the checkpoint and torch.Size([4, 512]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/Dissertation/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Kappa,Mae
1,1.394200,1.416132,0.242578,0.233283,0.001364,1.251687


/opt/anaconda3/envs/Dissertation/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


=== MobileNetV2 (google/mobilenet_v2_1.0_224) ===


...

=== MobileNetV3-Small (google/mobilenet_v1_1.0_224) ===


Some weights of MobileNetV1ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v1_1.0_224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1001, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([4]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/Dissertation/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Kappa,Mae
1,1.400300,1.422550,0.247301,0.232132,0.009193,1.294197


/opt/anaconda3/envs/Dissertation/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


,Model,Parameters(M),Size(MB),Speed(ms),accuracy,macro_f1,kappa,MAE
0,ResNet-18,11.18,42.7,8.7,0.2426,0.2333,0.0014,1.2517
1,MobileNetV2,2.23,8.7,7.8,0.2439,0.2266,-0.0183,1.2794
2,MobileNetV3-Small,3.21,12.4,3.8,0.2473,0.2321,0.0092,1.2942


In [37]:
for name, cm in cms.items():
    print(f'\n=== {name} (column=real, raw=predicted) ===')
    print(cm)


=== ResNet-18 (column=real, raw=predicted) ===
[[227 115 275 136]
 [201  97 242 131]
 [228 133 267 153]
 [217 125 289 128]]

=== MobileNetV2 (column=real, raw=predicted) ===
[[106  87 282 278]
 [111  73 234 253]
 [129  84 279 289]
 [124  83 287 265]]

=== MobileNetV3-Small (column=real, raw=predicted) ===
[[319 107 216 111]
 [300  95 181  95]
 [328  95 217 141]
 [319 106 232 102]]
